In [1]:
import pandas as pd
import xarray as xr
import numpy as np

from metpy.calc import wind_direction, wind_speed, wind_components
from metpy.units import units

data_dir = '../data_out/'
fig_png_dir = '../figures_png/'
fig_pdf_dir = '../figures_pdf/'

In [2]:
def get_df_from_nc(file, var):
    ds = xr.open_dataset(file)
    df = pd.DataFrame(ds[var].values, columns=ds[ds[var].dims[1]].values.astype(int), index=pd.to_datetime(ds.time,utc=True))
    return df

def resample_df(wspd, wdir, res):
    df = pd.DataFrame(index=wspd.index)

    if res is None:
        df['wdir'] = wdir
        df['wspd'] = wspd
    else:
        df['u'],df['v'] = wind_components(wspd.values * units('m/s'), wdir.values * units.deg)
        df = df.resample(res, origin='end_day').mean()
        df['wdir'] = wind_direction(df['u'].values * units('m/s'), df['v'].values * units('m/s'))
        df['wspd'] = wind_speed(df['u'].values * units('m/s'), df['v'].values * units('m/s'))
    return df

def get_value_pairs(h_obs, h_icon, res):
    df_obs  = resample_df(df_wspd[h_obs], df_wdir[h_obs], res).add_suffix('_obs')
    df_icon = resample_df(df_i_wspd[h_icon], df_i_wdir[h_icon], res).add_suffix('_icon')

    return pd.concat([df_obs, df_icon], axis=1).dropna()

def get_stats(df, min_wspd=0.0, time_limits=None):
    if min_wspd > 0.0:
        df = df[df['wspd_icon'] >= min_wspd]  # only consider records with minimum observed wind speed

    if time_limits is not None:
        df = df.loc[time_limits[0]:time_limits[1]]

    df['d_wdir'] = pd.concat([abs(df['wdir_icon'] - df['wdir_obs']),
                              abs(df['wdir_icon'] - df['wdir_obs'] - 360),
                              abs(df['wdir_icon'] - df['wdir_obs'] + 360)], axis=1).min(axis=1)

    stats = {'n':len(df),
             'wdir_rmse': np.sqrt((df['d_wdir']**2).mean()).round(2),
             'wdir_rmse_r': (np.sqrt((df['d_wdir']**2).mean()) / 180 * 100).round(2),
             'wdir_mae': df['d_wdir'].mean().round(2),
             'wspd_rmse': np.sqrt(((df['wspd_icon'] - df['wspd_obs'])**2).mean()).round(2),
             'wspd_mae': abs(df['wspd_icon'] - df['wspd_obs']).mean().round(2),
             'wspd_mbe': (df['wspd_icon'] - df['wspd_obs']).mean().round(2)
             }
    return stats

## Script settings

In [3]:
scope ='H3' # campaign: 'H2' or 'H3'

res = '3h'

In [4]:
# do scope specific settings
scope_name = 'HEFEX III'



dt_from,dt_to = '2025-08-06', '2025-08-31 23:59'
# dt_from,dt_to = '2025-08-10', '2025-08-10'
# dt_from,dt_to = '2025-08-15', '2025-08-15'

file_obs  = f'HEFEX3__Obs_WindRanger_L1__avg30min_20250807-20250831.nc'
file_icon = f'HEFEX3__ICON_v370_2030_cid35704_75ml__avg30min_20250805-20250905.nc'

## Get data (ICON & Obs) and resample to daily

The data used for this figure is available at Zenodo:

HEFEX III: #ToDo

In [5]:
df_wdir = get_df_from_nc(data_dir + file_obs, f'wdir')
df_wspd = get_df_from_nc(data_dir + file_obs, f'wspd')

df_i_wdir = get_df_from_nc(data_dir + file_icon, f'wdir').iloc[:,-15:]
df_i_wspd = get_df_from_nc(data_dir + file_icon, f'wspd').iloc[:,-15:]

In [6]:
print(sorted(df_wdir.columns))
print(sorted(df_i_wdir.columns.astype(int)))

[7, 10, 15, 20, 30, 40, 50, 75, 100, 125, 150, 175, 200]
[5, 15, 26, 39, 53, 68, 84, 100, 117, 134, 152, 170, 189, 207, 226]


In [8]:
res = '3h'

min_wspd = 0.0
time_limits = None #['2025-08-10', '2025-08-10']

df = pd.DataFrame({
    '15m' : get_stats(get_value_pairs(15,15,res), min_wspd=min_wspd, time_limits=time_limits),
    '100m': get_stats(get_value_pairs(100,100,res), min_wspd=min_wspd, time_limits=time_limits),
    '150m': get_stats(get_value_pairs(150,152,res), min_wspd=min_wspd, time_limits=time_limits),
    '200m': get_stats(get_value_pairs(200,207,res), min_wspd=min_wspd, time_limits=time_limits),
})

df.round(2)

,15m,100m,150m,200m
n,189.00,174.00,171.00,168.00
wdir_rmse,64.66,73.13,69.84,79.58
wdir_rmse_r,35.92,40.63,38.80,44.21
wdir_mae,44.80,52.99,51.37,60.56
wspd_rmse,1.36,1.67,1.56,2.06
wspd_mae,1.06,1.18,1.14,1.44
wspd_mbe,-0.20,-0.39,-0.33,-0.55


In [9]:
latex_table = df.to_latex(float_format="%.2f")
print(latex_table)

\begin{tabular}{lrrrr}
\toprule
 & 15m & 100m & 150m & 200m \\
\midrule
n & 189.00 & 174.00 & 171.00 & 168.00 \\
wdir_rmse & 64.66 & 73.13 & 69.84 & 79.58 \\
wdir_rmse_r & 35.92 & 40.63 & 38.80 & 44.21 \\
wdir_mae & 44.80 & 52.99 & 51.37 & 60.56 \\
wspd_rmse & 1.36 & 1.67 & 1.56 & 2.06 \\
wspd_mae & 1.06 & 1.18 & 1.14 & 1.44 \\
wspd_mbe & -0.20 & -0.39 & -0.33 & -0.55 \\
\bottomrule
\end{tabular}

